# 19 · Gray-Box AIA72 Multimodal OOD / Applicability Audit

Post-hoc Phase C audit. No fitting, recalibration, threshold tuning, policy selection, or redesign. SHARP OOD uses the previously declared squared Mahalanobis distance (Ledoit–Wolf fit on 2010–2013 transformed 3×16 SHARP histories) and the already-frozen q99 threshold 379.55586300796773. AIA input status/seed spread are applicability proxies only; no AIA embedding-space OOD claim is made.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib,json,numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.covariance import LedoitWolf
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp,spearmanr
ROOT=Path('/home/abmoses2000'); Q99=379.55586300796773; SEEDS=[17,29,43]
MASTER=ROOT/'graybox_aia72_master_predictions_v2_20261005/graybox_aia72_master_predictions.csv.gz'; M_SHA='17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46'
SHARP=ROOT/'sharp72_predictions_frozen_20261002.csv.gz'; S_SHA='7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b'
MOND=ROOT/'graybox_aia72_conformal_mondrian_v2_20261005/mondrian_thresholds.csv'; ALARM=ROOT/'graybox_aia72_alarm_and_primary_policy_freeze_v1_20261005/alarm_thresholds.csv'
OUT=ROOT/'graybox_aia72_multimodal_ood_applicability_v1_20261006'
if OUT.exists(): raise SystemExit(f'Refusing to overwrite {OUT}')
(OUT/'figures').mkdir(parents=True)
def sha(p):
 h=hashlib.sha256(); f=open(p,'rb')
 for c in iter(lambda:f.read(4*1024*1024),b''): h.update(c)
 f.close(); return h.hexdigest()
def req(c,m):
 if not c: raise ValueError(m)
req(sha(MASTER)==M_SHA,'Master SHA changed'); req(sha(SHARP)==S_SHA,'SHARP SHA changed')
a=pd.read_csv(MASTER,low_memory=False); s=pd.read_csv(SHARP,low_memory=False)
a.forecast_case_id=a.forecast_case_id.astype(str); s.forecast_case_id=s.forecast_case_id.astype(str)
m=a.merge(s,on='forecast_case_id',suffixes=('_aia','_sharp'),validate='one_to_one'); req(len(m)==len(a),'Alignment loss')
la='label_aia' if 'label_aia' in m else 'label'; ls='label_sharp' if 'label_sharp' in m else la
m['y']=pd.to_numeric(m[la]).astype(int); req(np.array_equal(m.y,pd.to_numeric(m[ls]).astype(int)),'Label mismatch')
issue=pd.Series(pd.NaT,index=m.index,dtype='datetime64[ns, UTC]')
for c in [x for x in ['issue_utc_aia','issue_utc_sharp','issue_utc','T_REC_dt_aia','T_REC_dt_sharp','T_REC_dt','T_REC_aia','T_REC_sharp','T_REC'] if x in m]:
 z=pd.to_datetime(m[c],utc=True,errors='coerce'); take=issue.isna()&z.notna(); issue.loc[take]=z.loc[take]
z=pd.to_datetime(m.forecast_case_id.str.extract(r':(\d{8}_\d{4})_',expand=False),format='%Y%m%d_%H%M',utc=True,errors='coerce'); take=issue.isna()&z.notna(); issue.loc[take]=z.loc[take]
req(issue.notna().all(),'Unresolved issue times'); m['year']=issue.dt.year.astype(int)
rc='role_aia' if 'role_aia' in m else 'role'; m['role_final']=m[rc].astype(str)
m['sharp_p']=pd.to_numeric(m.probability_gru_mean,errors='coerce'); m['aia_p']=pd.to_numeric(m.probability,errors='coerce'); m['fusion_p']=.5*(m.sharp_p+m.aia_p)
def scol(side):
 return [f'probability_seed_{z}_{side}' if f'probability_seed_{z}_{side}' in m else f'probability_seed_{z}' for z in SEEDS]
AS=m[scol('aia')].astype(float).to_numpy(); SS=m[scol('sharp')].astype(float).to_numpy(); FS=.5*(AS+SS)
m['aia_seed_std']=AS.std(1,ddof=1); m['fusion_seed_std']=FS.std(1,ddof=1); m['gap']=abs(m.sharp_p-m.aia_p)
q=np.clip(m.fusion_p.to_numpy(float),1e-12,1-1e-12); m['entropy']=-(q*np.log(q)+(1-q)*np.log(1-q))
def rg(r):
 return 'cycle25_2021_2025_diagnostic' if r.role_final=='retrospective_cycle25' else ('supplementary_2026_locked' if r.role_final=='supplementary_2026' else ('cycle24_earlier_development' if r.year<=2019 else 'other_or_gap'))
m['regime']=m.apply(rg,axis=1)
# Reuse frozen feature distance if present; otherwise reconstruct from pinned tensor/transform.
if 'feature_distance_squared' in s:
 fd=s[['forecast_case_id','feature_distance_squared']].copy(); distance_source='frozen_column'
else:
 ds=ROOT/'gray-box-solar-flare-forecasting/data/processed/training_snapshot_20261002/gray_box_aligned_v1'; tr=ROOT/'gray-box-solar-flare-forecasting/outputs/sharp72_notebook_v1_20261002T080801721986Z/transform.npz'
 if not (ds/'sharp.npy').is_file():
  hits=[p for p in ROOT.glob('**/training_snapshot_20261002/gray_box_aligned_v1') if (p/'sharp.npy').is_file()]; req(len(hits)==1,f'Dataset hits={hits}'); ds=hits[0]
 if not tr.is_file():
  hits=list(ROOT.glob('**/sharp72_notebook_v1_20261002T080801721986Z/transform.npz')); req(len(hits)==1,f'Transform hits={hits}'); tr=hits[0]
 raw=np.load(ds/'sharp.npy',allow_pickle=False); idx=pd.read_csv(ds/'input_index.csv.gz'); req(len(raw)==len(idx),'Tensor/index mismatch')
 with np.load(tr,allow_pickle=False) as t: norm=((np.sign(raw)*np.log1p(abs(raw))-t['mean'])/t['std']).astype(np.float32)
 rows=pd.to_numeric(s.tensor_row).astype(int).to_numpy(); X=norm[rows].reshape(len(s),-1).astype(float); fit=s.role.astype(str).eq('train').to_numpy(); req(fit.sum()==25586,'Train support changed')
 est=LedoitWolf().fit(X[fit]); req(abs(float(est.shrinkage_)-0.0004973324331986849)<1e-12,'Covariance changed')
 fd=pd.DataFrame({'forecast_case_id':s.forecast_case_id,'feature_distance_squared':est.mahalanobis(X)}); distance_source=f'reconstructed:{ds}'
m=m.merge(fd,on='forecast_case_id',validate='one_to_one'); req(np.isfinite(m.feature_distance_squared).all(),'Missing distance'); m['ood_q99']=m.feature_distance_squared>Q99
mond=pd.read_csv(MOND); alarm=pd.read_csv(ALARM); ath={r.branch:float(r.threshold) for r in alarm.itertuples()}
for b,pcol in [('sharp','sharp_p'),('aia','aia_p'),('sharp_aia','fusion_p')]:
 z=mond[(mond.branch==b)&np.isclose(mond.alpha.astype(float),.1)]; req(len(z)==1,f'Missing {b} qhat'); q0,q1=float(z.iloc[0].qhat_class0),float(z.iloc[0].qhat_class1); p=m[pcol].to_numpy(float); y=m.y.to_numpy(int); inc0=p<=q0; inc1=(1-p)<=q1; m[b+'_covered']=np.where(y==1,inc1,inc0); m[b+'_err']=(p>=ath[b]).astype(int)!=y
status='input_status' if 'input_status' in m else ('input_status_aia' if 'input_status_aia' in m else None); m['aia_quality_ok']=True if status is None else m[status].astype(str).eq('ok')
rows=[]
for r,g in m.groupby('regime',sort=False):
 rows.append({'regime':r,'cases':len(g),'positives':int(g.y.sum()),'prevalence':g.y.mean(),'distance_mean':g.feature_distance_squared.mean(),'distance_median':g.feature_distance_squared.median(),'distance_p95':g.feature_distance_squared.quantile(.95),'ood_q99_rate':g.ood_q99.mean(),'aia_quality_failure_rate':(~g.aia_quality_ok).mean(),'aia_seed_std_mean':g.aia_seed_std.mean(),'fusion_seed_std_mean':g.fusion_seed_std.mean(),'gap_mean':g.gap.mean(),'entropy_mean':g.entropy.mean(),'sharp_error_rate':g.sharp_err.mean(),'fusion_error_rate':g.sharp_aia_err.mean(),'sharp_flare_coverage':g.loc[g.y.eq(1),'sharp_covered'].mean(),'fusion_flare_coverage':g.loc[g.y.eq(1),'sharp_aia_covered'].mean()})
reg=pd.DataFrame(rows); reg.to_csv(OUT/'regime_applicability_summary.csv',index=False)
fr=[]
for r,g in m.groupby('regime',sort=False):
  for flag,h in g.groupby('ood_q99'):
   flare=h.y.eq(1); fr.append({'regime':r,'ood_q99':bool(flag),'cases':len(h),'positives':int(flare.sum()),'sharp_error_rate':h.sharp_err.mean(),'fusion_error_rate':h.sharp_aia_err.mean(),'sharp_flare_coverage':h.loc[flare,'sharp_covered'].mean() if flare.any() else np.nan,'fusion_flare_coverage':h.loc[flare,'sharp_aia_covered'].mean() if flare.any() else np.nan,'distance_mean':h.feature_distance_squared.mean()})
flags=pd.DataFrame(fr); flags.to_csv(OUT/'sharp_q99_ood_flag_diagnostics.csv',index=False)
signals={'sharp_feature_distance':'feature_distance_squared','gap':'gap','fusion_seed_std':'fusion_seed_std','aia_seed_std':'aia_seed_std','fusion_entropy':'entropy'}; assoc=[]
for r,g in m.groupby('regime',sort=False):
  for sn,sc in signals.items():
   x=g[sc].to_numpy(float)
   for tn,tc in [('sharp_error','sharp_err'),('fusion_error','sharp_aia_err'),('sharp_coverage_failure','sharp_covered'),('fusion_coverage_failure','sharp_aia_covered')]:
    y=(~g[tc].astype(bool)).astype(int).to_numpy() if 'coverage' in tn else g[tc].astype(int).to_numpy(); auc=roc_auc_score(y,x) if len(np.unique(y))==2 else np.nan; rho,pv=spearmanr(x,y); assoc.append({'regime':r,'signal':sn,'target':tn,'roc_auc_for_failure':auc,'spearman_rho':rho,'spearman_p':pv})
pd.DataFrame(assoc).to_csv(OUT/'signal_failure_associations.csv',index=False)
base=m[m.regime.eq('cycle24_earlier_development')]; sh=[]
for r in ['cycle25_2021_2025_diagnostic','supplementary_2026_locked']:
  cur=m[m.regime.eq(r)]
  for sn,sc in signals.items():
   st,p=ks_2samp(base[sc],cur[sc]); sh.append({'reference':'cycle24_earlier_development','comparison':r,'signal':sn,'ks_statistic':st,'ks_p':p,'reference_mean':base[sc].mean(),'comparison_mean':cur[sc].mean()})
shift=pd.DataFrame(sh); shift.to_csv(OUT/'distribution_shift_tests.csv',index=False)
order=['cycle24_earlier_development','cycle25_2021_2025_diagnostic','supplementary_2026_locked']; pr=reg.set_index('regime').loc[order]
for cols,title,name in [(['ood_q99_rate'],'Frozen SHARP q99 feature-distance flag rate','fig1_q99_rate.png'),(['distance_mean','distance_median'],'SHARP feature-space distance','fig2_distance.png'),(['gap_mean','fusion_seed_std_mean','entropy_mean'],'Predictive agreement/confidence signals','fig3_trust_signals.png'),(['sharp_error_rate','fusion_error_rate'],'Alarm error rate','fig4_error_rate.png')]:
 ax=pr[cols].plot(kind='bar',figsize=(9,5),legend=len(cols)>1); ax.set_title(title); ax.set_xlabel(''); plt.xticks(rotation=15,ha='right'); plt.tight_layout(); plt.savefig(OUT/'figures'/name,dpi=220); plt.close()
protocol={'status':'POSTHOC_MULTIMODAL_OOD_APPLICABILITY_AUDIT_COMPLETE','created_utc':datetime.now(timezone.utc).isoformat(),'new_model_fitting':False,'new_threshold_tuning':False,'new_policy_selection':False,'new_policy_redesign':False,'sharp_q99_threshold':Q99,'distance_source':distance_source,'aia_feature_space_ood_available':False,'aia_applicability_note':'input status and seed spread only; no image-embedding OOD claim','source_sha256':{'master':M_SHA,'sharp':S_SHA}}
(OUT/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')
print('===== MULTIMODAL OOD / APPLICABILITY AUDIT COMPLETE ====='); print('Distance source:',distance_source); print(reg.round(6).to_string(index=False)); print('\nQ99 diagnostic:\n',flags.round(6).to_string(index=False)); print('\nShift tests:\n',shift.round(6).to_string(index=False)); print('\nOutputs:',OUT)


## Interpretation guardrails

A high SHARP distance is evidence of deviation from the frozen training feature distribution, not proof a forecast is wrong. The q99 cutoff is inherited and not optimized here. AIA seed spread/input quality are proxies only. Error/coverage associations are post-hoc diagnostics, not policy selection.
